# Desc Bike Classic ML Data Prep
Build the station-level model dataframe and spatial fold table for the description-based Wikidata semantics experiment.


In [ ]:
# Imports and fixed project paths.
from pathlib import Path
import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from node2vec import Node2Vec
from sklearn.cluster import KMeans
PROJECT_ROOT = Path("/home/najla/dev/najla-msc/bikeshare")
DATA_DIR = PROJECT_ROOT / "data"
BIKE_DIR = DATA_DIR / "processed/bike_ridership"
GRAPH_DIR = DATA_DIR / "processed/graph"
DESC_SEMANTIC_FILE = (
    GRAPH_DIR
    / "desc_semantic/desc_all_wikidata_minilm_embeddings.parquet"
)
OUTPUT_DIR = DATA_DIR / "processed/model_df/desc_semantics"
MODEL_DF_FILE = OUTPUT_DIR / "desc_df_2targets_txtTokens_graphFeat.parquet"
FOLD_ASSIGNMENTS_FILE = (
    OUTPUT_DIR
    / "desc_folds_of_2targets_txtTokens_graphFeat.parquet"
)
# This disconnected island node has no graph neighbours in the Toronto graph.
ISLAND_LOC_ID = "5350002.00"


In [ ]:
# Load static CT features and remove the disconnected island node.
static_features = pd.read_parquet(
    BIKE_DIR / "static_features_5yrs_12pm.parquet"
)
static_features["loc_id"] = static_features["loc_id"].astype(str)
static_features = (
    static_features
    .loc[static_features["loc_id"] != ISLAND_LOC_ID]
    .reset_index(drop=True)
)
# Load daily dynamic features and keep the weather name used downstream.
dynamic_features = pd.read_parquet(
    BIKE_DIR / "dynamic_features_5yrs_12pm.parquet"
)
dynamic_features["date"] = pd.to_datetime(dynamic_features["date"])
dynamic_features = dynamic_features.rename(
    columns={"weather_category": "Main_Weather_Category"}
)
# Load description-based semantic features.
# This file already contains one row per graph node, MiniLM embeddings,
# attraction_count, and attraction_missing_flag.
desc_semantic = pd.read_parquet(DESC_SEMANTIC_FILE)
desc_semantic["loc_id"] = desc_semantic["loc_id"].astype(str)
wiki_embedding_cols = [
    col
    for col in desc_semantic.columns
    if col.startswith("wiki_emb_")
]
desc_semantic = (
    desc_semantic[
        [
            "loc_id",
            *wiki_embedding_cols,
            "attraction_count",
            "attraction_missing_flag",
        ]
    ]
    .drop_duplicates(subset="loc_id")
    .reset_index(drop=True)
)
# Keep the missingness convention used by the previous pipeline.
desc_semantic["attraction_missing_flag"] = (
    desc_semantic["attraction_missing_flag"]
    .fillna(1)
    .astype(int)
)
desc_semantic["attraction_count"] = (
    desc_semantic["attraction_count"]
    .fillna(-1)
)
desc_semantic.loc[
    desc_semantic["attraction_missing_flag"].eq(1),
    "attraction_count",
] = -1
# Keep a simple text marker so downstream code that expects wiki_items_text
# can still distinguish real description semantics from missing nodes.
desc_semantic["wiki_items_text"] = np.where(
    desc_semantic["attraction_missing_flag"].eq(1),
    "no_wikidata",
    "desc_wikidata",
)
# The semantic file must cover every static node that remains after filtering.
missing_semantic_nodes = sorted(
    set(static_features["loc_id"])
    - set(desc_semantic["loc_id"])
)
assert not missing_semantic_nodes, missing_semantic_nodes[:10]
print("Static features:", static_features.shape)
print("Dynamic features:", dynamic_features.shape)
print("Description semantic features:", desc_semantic.shape)
print("Wiki embedding columns:", len(wiki_embedding_cols))


In [ ]:
# Build Node2Vec graph features from the processed CT edge table.
edges = pd.read_parquet(GRAPH_DIR / "edges.parquet").reset_index()
edges["level_0"] = edges["level_0"].astype(str)
edges["level_1"] = edges["level_1"].astype(str)
# Convert edge distance into a proximity weight for random walks.
edges["distance_m"] = edges["weight"]
edges["affinity_weight"] = 1 / (edges["distance_m"] + 1e-6)
graph = nx.from_pandas_edgelist(
    edges,
    source="level_0",
    target="level_1",
    edge_attr="affinity_weight",
)
# Keep the original random-walk settings from the station-level notebook.
node2vec = Node2Vec(
    graph,
    dimensions=16,
    walk_length=10,
    num_walks=50,
    p=1.0,
    q=2.0,
    weight_key="affinity_weight",
    workers=4,
    seed=42,
)
node2vec_model = node2vec.fit(
    window=5,
    min_count=1,
)
node2vec_rows = []
for node in graph.nodes():
    vector = node2vec_model.wv[str(node)]
    row = {"loc_id": str(node)}
    for i, value in enumerate(vector):
        row[f"node2vec_{i}"] = value
    node2vec_rows.append(row)
node2vec_features = pd.DataFrame(node2vec_rows)
node2vec_cols = [
    col
    for col in node2vec_features.columns
    if col.startswith("node2vec_")
]
print("Node2Vec features:", node2vec_features.shape)


In [ ]:
# Read station inflow/outflow targets and align them to the same schema.
inflow = pd.read_parquet(BIKE_DIR / "inflow_5yrs_12pm.parquet")
outflow = pd.read_parquet(BIKE_DIR / "outflow_5yrs_12pm.parquet")
inflow_station = inflow.rename(
    columns={
        "end_loc_id": "loc_id",
        "end_station_name": "station_name",
        "end_capacity_avg": "station_capacity_avg",
        "end_trips_count": "inflow_count",
    }
)
outflow_station = outflow.rename(
    columns={
        "start_loc_id": "loc_id",
        "start_station_name": "station_name",
        "start_capacity_avg": "station_capacity_avg",
        "start_trips_count": "outflow_count",
    }
)
flow_station = pd.concat(
    [
        inflow_station,
        outflow_station,
    ],
    ignore_index=True,
)
flow_station["loc_id"] = flow_station["loc_id"].astype(str)
flow_station["date"] = pd.to_datetime(flow_station["date"])
flow_station = (
    flow_station
    .groupby(
        [
            "loc_id",
            "date",
            "station_name",
        ],
        as_index=False,
    )
    .agg(
        station_capacity_avg=("station_capacity_avg", "mean"),
        inflow_count=("inflow_count", "sum"),
        outflow_count=("outflow_count", "sum"),
    )
)
flow_station["inflow_count"] = flow_station["inflow_count"].fillna(0)
flow_station["outflow_count"] = flow_station["outflow_count"].fillna(0)
print("Station flow table:", flow_station.shape)


In [ ]:
# Merge static, station-flow, description-semantic, graph, and dynamic features.
static_join = static_features.copy()
semantic_join = desc_semantic.copy()
graph_join = node2vec_features.copy()
dynamic_join = dynamic_features.copy()
for table in [
    static_join,
    semantic_join,
    graph_join,
]:
    table["loc_id"] = table["loc_id"].astype(str)
df_prep = (
    static_join
    .merge(
        flow_station,
        on="loc_id",
        how="left",
        validate="one_to_many",
    )
    .merge(
        semantic_join,
        on="loc_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        graph_join,
        on="loc_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        dynamic_join,
        on="date",
        how="left",
        validate="many_to_one",
    )
    .drop_duplicates()
)
# Keep rows where at least one station target exists.
df_prep = df_prep.loc[
    ~(
        df_prep["inflow_count"].isna()
        & df_prep["outflow_count"].isna()
    )
].copy()
print("Prepared dataframe before cleanup:", df_prep.shape)


In [ ]:
# Final cleanup and spatial grouping.
df = df_prep.copy()
# Drop rows without graph embeddings.
df = df.dropna(subset=node2vec_cols).copy()
# Enforce the description-semantic missingness convention after the merge.
df["attraction_missing_flag"] = (
    df["attraction_missing_flag"]
    .fillna(1)
    .astype(int)
)
df["attraction_count"] = df["attraction_count"].fillna(-1)
df.loc[
    df["attraction_missing_flag"].eq(1),
    "attraction_count",
] = -1
df["wiki_items_text"] = np.where(
    df["attraction_missing_flag"].eq(1),
    "no_wikidata",
    "desc_wikidata",
)
# Keep the existing station-count and target defaults.
df["total_stations"] = df["total_stations"].fillna(
    df["end_stations_count"]
)
df["inflow_count"] = df["inflow_count"].fillna(0)
df["outflow_count"] = df["outflow_count"].fillna(0)
# Cluster CT centroids into spatial groups for spatial cross-validation.
df["loc_id_key"] = df["loc_id"].astype(float).map(
    lambda value: f"{value:010.2f}"
)
ct_centers = (
    df[
        [
            "loc_id",
            "loc_id_key",
            "lat",
            "lon",
        ]
    ]
    .dropna(subset=["lat", "lon"])
    .drop_duplicates(subset="loc_id_key")
    .copy()
)
ct_centers = gpd.GeoDataFrame(
    ct_centers,
    geometry=gpd.points_from_xy(
        ct_centers["lon"],
        ct_centers["lat"],
    ),
    crs="EPSG:4326",
).to_crs(epsg=26917)
ct_centers["x"] = ct_centers.geometry.x
ct_centers["y"] = ct_centers.geometry.y
kmeans = KMeans(
    n_clusters=20,
    random_state=42,
    n_init=10,
)
ct_centers["spatial_group"] = kmeans.fit_predict(
    ct_centers[
        [
            "x",
            "y",
        ]
    ]
)
df = df.merge(
    ct_centers[
        [
            "loc_id_key",
            "spatial_group",
        ]
    ],
    on="loc_id_key",
    how="left",
)
# Remove helper columns that are not model inputs.
df = df.drop(
    columns=[
        "start_stations_count",
        "end_stations_count",
        "loc_id_key",
    ],
    errors="ignore",
)
assert df["spatial_group"].notna().all()
assert df["attraction_count"].notna().all()
assert df["attraction_missing_flag"].notna().all()
print("Final model dataframe:", df.shape)
display(
    df[
        [
            "loc_id",
            "date",
            "station_name",
            "inflow_count",
            "outflow_count",
            "attraction_count",
            "attraction_missing_flag",
            "spatial_group",
        ]
    ].head()
)


In [ ]:
# Create row-balanced spatial folds from the spatial groups.
def make_row_balanced_spatial_folds(
    df,
    group_col="spatial_group",
    n_folds=10,
    n_test_groups=3,
    n_val_groups=3,
    seed=42,
    n_candidates=20000,
):
    group_counts = df.groupby(group_col).size()
    groups = group_counts.index.to_numpy()
    total_rows = len(df)
    fold_specs = []
    seen_splits = set()
    rng = np.random.default_rng(seed)
    for fold_id in range(n_folds):
        best = None
        best_score = np.inf
        for _ in range(n_candidates):
            shuffled = rng.permutation(groups)
            test_groups = tuple(sorted(shuffled[:n_test_groups]))
            val_groups = tuple(
                sorted(shuffled[n_test_groups:n_test_groups + n_val_groups])
            )
            train_groups = tuple(
                sorted(shuffled[n_test_groups + n_val_groups:])
            )
            split_key = (
                train_groups,
                val_groups,
                test_groups,
            )
            if split_key in seen_splits:
                continue
            train_rows = group_counts.loc[list(train_groups)].sum()
            val_rows = group_counts.loc[list(val_groups)].sum()
            test_rows = group_counts.loc[list(test_groups)].sum()
            train_pct = train_rows / total_rows
            val_pct = val_rows / total_rows
            test_pct = test_rows / total_rows
            score = (
                abs(train_pct - 0.70)
                + abs(val_pct - 0.15)
                + abs(test_pct - 0.15)
            )
            if score < best_score:
                best_score = score
                best = {
                    "fold_id": fold_id,
                    "train_groups": train_groups,
                    "val_groups": val_groups,
                    "test_groups": test_groups,
                    "train_rows": train_rows,
                    "val_rows": val_rows,
                    "test_rows": test_rows,
                    "train_pct": train_pct,
                    "val_pct": val_pct,
                    "test_pct": test_pct,
                }
        if best is None:
            raise RuntimeError(
                f"Could not build a unique split for fold {fold_id}."
            )
        seen_splits.add(
            (
                best["train_groups"],
                best["val_groups"],
                best["test_groups"],
            )
        )
        fold_specs.append(best)
    return fold_specs
fold_specs = make_row_balanced_spatial_folds(
    df,
    group_col="spatial_group",
    n_folds=10,
    n_test_groups=3,
    n_val_groups=3,
    seed=42,
)
fold_assignment_rows = []
for spec in fold_specs:
    for split_name, groups in [
        ("train", spec["train_groups"]),
        ("val", spec["val_groups"]),
        ("test", spec["test_groups"]),
    ]:
        for spatial_group in groups:
            fold_assignment_rows.append(
                {
                    "fold_id": spec["fold_id"],
                    "spatial_group": spatial_group,
                    "split": split_name,
                }
            )
fold_assignments_df = (
    pd.DataFrame(fold_assignment_rows)
    .sort_values(
        [
            "fold_id",
            "split",
            "spatial_group",
        ]
    )
    .reset_index(drop=True)
)
print("Fold assignments:", fold_assignments_df.shape)
display(fold_assignments_df.head(20))


In [ ]:
# Save the description-semantic model dataframe and fold design.
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
df.to_parquet(
    MODEL_DF_FILE,
    index=False,
)
fold_assignments_df.to_parquet(
    FOLD_ASSIGNMENTS_FILE,
    index=False,
)
print("Saved model dataframe:", MODEL_DF_FILE)
print("Saved fold assignments:", FOLD_ASSIGNMENTS_FILE)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Folds:", fold_assignments_df["fold_id"].nunique())
